In [1]:
import pandas as pd
import numpy as np


In [3]:
dataset_raw = pd.read_csv('../Data/Product_Normalization_GRI.csv')
dataset_raw.head()

,Room Description,Guest Room Info
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room


In [4]:
dataset_raw.describe()

,Room Description,Guest Room Info
count,35000,34999
unique,31237,35
top,2X POINTS PACKAGE|2 QUEEN BEDS STUDIO NONSMOKI...,Accessible Room
freq,49,1000


In [5]:
dataset_raw.isnull().sum()

Room Description    0
Guest Room Info     1
dtype: int64

In [6]:
#Dataset has empty value in Bungalow section
dataset_raw.fillna('Bungalow', inplace=True)

In [7]:
len(dataset_raw)

35000

In [8]:
dataset_raw

,Room Description,Guest Room Info
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room
...,...,...
34995,CWT INCLUDES WIFI USE OF|FITNESS AND WELLNESS ...,Run of the House
34996,STAY LONGER SAVE|SPECIALTY ROOM WHEN YOU ARRIV...,Run of the House
34997,CWT|SPECIALTY ROOM WHEN YOU ARRIVE AT THE HOTE...,Run of the House
34998,"BOOK NOW, PAY LATER|DELUXE ROOM AN UPGRADE FRO...",Run of the House


In [20]:
deduplicated_data = dataset_raw.drop_duplicates(subset=['Room Description']).reset_index(drop=True)
print(f"Original rows: {len(dataset_raw)}")
print(f"After deduplication: {len(deduplicated_data)}")


Original rows: 35000
After deduplication: 31237


In [21]:
def clean_text(texts):
    """Clean and standardize text descriptions"""
    cleaned_texts = []
    for text in texts:
        # Replace specific compound term
        compound_terms = {
            'bed room': 'bedroom'
        }
        
        for term, replacement in compound_terms.items():
            text = text.replace(term, replacement)
            
        cleaned_texts.append(text)
    
    return cleaned_texts

In [22]:
deduplicated_data['Room Description'] = clean_text(deduplicated_data['Room Description'])

In [46]:
deduplicated_data['Room Description'][2110]

'ROMANCE PACKAGE|1 BDRM EXECUTIVE SUITE-1 QUEEN BED 2 FLAT SCREEN TVS-BROADBAND CHRG APPLY'

## Detecting potential abbreviations in Descriptions ##

In [23]:
from collections import Counter
import re
import pandas as pd
from tqdm import tqdm

def detect_potential_abbreviations(descriptions):
    """Extract and analyze potential abbreviations from descriptions, based on the token length and frequency"""
    # Extract all word-like tokens
    all_tokens = []
    for desc in tqdm(descriptions, desc="Processing descriptions"):
        if pd.isna(desc):
            continue
        # Extract tokens that look like abbreviations
        tokens = re.findall(r'\b[A-Z0-9/]{2,4}\b', str(desc).upper())
        all_tokens.extend(tokens)
    
    # Count frequencies
    token_counts = Counter(all_tokens)
    
    # Filter and sort by frequency
    potential_abbrevs = {
        token: count for token, count in token_counts.items()
        if count > 10  # Appear in at least 10 descriptions
    }
    
    # Sort by frequency
    sorted_abbrevs = dict(sorted(potential_abbrevs.items(), 
                                key=lambda x: x[1], 
                                reverse=True))
    
    return sorted_abbrevs

# Test the function
potential_abbrevs = detect_potential_abbreviations(deduplicated_data['Room Description'])

# Display results
print("\nTop 20 potential abbreviations:")
for abbrev, count in list(potential_abbrevs.items())[:20]:
    print(f"{abbrev}: {count} occurrences")

Processing descriptions: 100%|██████████| 31237/31237 [00:00<00:00, 254401.87it/s]


Top 20 potential abbreviations:
ROOM: 17834 occurrences
RATE: 16601 occurrences
KING: 16490 occurrences
BED: 13944 occurrences
WIFI: 10266 occurrences
FREE: 7641 occurrences
AND: 7380 occurrences
VIEW: 7132 occurrences
WITH: 6564 occurrences
BEST: 4462 occurrences
IN: 3364 occurrences
BEDS: 3193 occurrences
TO: 3192 occurrences
TV: 3078 occurrences
SQ: 3078 occurrences
SOFA: 3053 occurrences
FT: 2777 occurrences
CITY: 2774 occurrences
OR: 2446 occurrences
TWIN: 2353 occurrences


In [47]:
pd.DataFrame(potential_abbrevs, index = [0]).T

,0
ROOM,17834
RATE,16601
KING,16490
BED,13944
WIFI,10266
...,...
FCA,11
770,11
VERY,11
SODA,11


In [48]:
pd.DataFrame(potential_abbrevs, index = [0]).T.to_excel('potential_abbrevs.xlsx', index=True)

## Expanding Common Abbreviations to Full forms ##

In [53]:
import re

def expand_abbreviations(text):
    """Expand common hotel abbreviations to full form"""
    
    # Common hotel abbreviations
    abbrev_dict = {
        # Room Types
        'RM': 'ROOM',
        'STE': 'SUITE',
        'STD': 'STANDARD',
        'DLX': 'DELUXE',
        'EXEC': 'EXECUTIVE',
        
        # Bed Types
        'KG': 'KING',
        'QN': 'QUEEN',
        'DBL': 'DOUBLE',
        'TWN': 'TWIN',
        
        # Accessibility
        'ADA': 'ACCESSIBLE',
        'ACCESS': 'ACCESSIBLE',
        'ACESS': 'ACCESSIBLE',
        'HEAR': 'HEARING',
        'ACCS': 'ACCESSIBLE',
        
        # View Types
        'MTN': 'MOUNTAIN',
        'OCN': 'OCEAN',
        'VW': 'VIEW',
        
        # Amenities
        'NOSMOK': 'NON SMOKING',
        'NSMK': 'NON SMOKING',
        'NS': 'NON SMOKING',
        'SMKG': 'SMOKING',
        'W/': 'WITH',
        'W': 'WITH',
        'W/O': 'WITHOUT',
        'BAL': 'BALCONY',
        'KITCH': 'KITCHEN',
        'KTCH': 'KITCHEN',
        
        # Common Words
        'FLR': 'FLOOR',
        'LVL': 'LEVEL',
        'BLDG': 'BUILDING',
        'BR': 'BEDROOM',
        'BDRM': 'BEDROOM',
        'BEDRM': 'BEDROOM'

    }
    
    # Sort by length (longest first) to avoid partial replacements
    sorted_abbrev = sorted(abbrev_dict.items(), key=lambda x: len(x[0]), reverse=True)
    
    # Replace abbreviations
    text = text.upper()  # Convert to uppercase for consistency
    for abbrev, full_form in sorted_abbrev:
        # Add word boundaries to avoid partial word matches
        pattern = r'\b' + re.escape(abbrev) + r'\b'
        text = re.sub(pattern, full_form, text)
    
    return text


In [54]:

# Example usage:
sample_texts = [
    "DLX BR KG RM W/ ALARM OCN VW",
    "EXEC STE W/ KITCH",
    "ADA QN RM W/ ROLL IN SHWR",
    "STD DBL NS W/ MTN VW"
]

for text in sample_texts:
    expanded = expand_abbreviations(text)
    print(f"Original: {text}")
    print(f"Expanded: {expanded}\n")

Original: DLX BR KG RM W/ ALARM OCN VW
Expanded: DELUXE BEDROOM KING ROOM WITH/ ALARM OCEAN VIEW

Original: EXEC STE W/ KITCH
Expanded: EXECUTIVE SUITE WITH/ KITCHEN

Original: ADA QN RM W/ ROLL IN SHWR
Expanded: ACCESSIBLE QUEEN ROOM WITH/ ROLL IN SHWR

Original: STD DBL NS W/ MTN VW
Expanded: STANDARD DOUBLE NON SMOKING WITH/ MOUNTAIN VIEW



In [55]:
deduplicated_data['Room Description Expanded'] = deduplicated_data['Room Description'].apply(expand_abbreviations)
deduplicated_data.head()


,Room Description,Guest Room Info,Room Description Expanded
0,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...,Accessible Room,FLEXIBLE RATE|2QUEEN HEARING ACCESSIBLE STUDIO...
1,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,2X POINTS PACKAGE|2QUEEN HEARING ACCESSIBLE ST...
2,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...,Accessible Room,PARK AND STAY AAA|2QUEEN HEARING ACCESSIBLE ST...
3,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...,Accessible Room,PARK AND STAY AARP|2QUEEN HEARING ACCESSIBLE S...
4,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...,Accessible Room,PARK AND GO|2QUEEN HEARING ACCESSIBLE STUDIO S...


In [56]:
deduplicated_data.to_csv('../Data/Product_Normalization_GRI_Expanded.csv')